# Biomass workflow (ESA MAAP)
The logic lives in the `biomass` package (`src/biomass`). Install once with `pip install -e ".[geo,dev]"`, edit only the configuration cell and run top to bottom.

In [ ]:
from pathlib import Path
from biomass import (BiomassClient, SearchQuery, Downloader, items_table, assets_table,
                     footprint_map, inspect_raster, plot_raster, raster_map)

# --- Configuration ---
QUERY = SearchQuery(
    collection="BiomassLevel1a",
    product_type="S1_SCS__1S",                 # None to skip filtering
    date_start="2026-01-01",
    date_end="2026-07-01",
    bbox=[-81.00, -20.00, -35.00, 10.00],      # [min_lon, min_lat, max_lon, max_lat]
    aoi_file=None,                             # e.g. "my_area.geojson" (overrides bbox)
    extra_filters={},                          # e.g. {"sat:orbit_state": "DESCENDING"}
    max_items=10,
)
ASSETS_TO_DOWNLOAD = ["quicklook_jpg", "enclosure_i_abs_tiff"]
OUTPUT_DIR = Path("../data")

client = BiomassClient()
client.auth.access_token(); print("Token OK")

## Explore

In [ ]:
client.collections()

In [ ]:
client.queryables(QUERY.collection)

## Search

In [ ]:
items = client.search(QUERY)
items_table(items)

In [ ]:
footprint_map(items, QUERY)

## Download

In [ ]:
assets_table(items[0])

In [ ]:
files = Downloader(client.auth, OUTPUT_DIR).download(items, ASSETS_TO_DOWNLOAD)   # test first with items[:1]

## Inspect

In [ ]:
inspect_raster(next(OUTPUT_DIR.rglob("*.tiff")))

## Visualize
`plot_raster` shows every polarisation of one file (dB, radar geometry). `raster_map` lays the downloaded files over a map: `band="rgb"` is a colour composite (R=HH, G=HV, B=VV), or pass `"HH"`, `"HV"`, `"VH"`, `"VV"` for a single polarisation.

In [ ]:
tiffs = sorted(OUTPUT_DIR.rglob("*.tiff"))
plot_raster(tiffs[0]);

In [ ]:
raster_map(tiffs)   # one layer per file; toggle them with the control in the top-right corner